In [1]:
import numpy as np
import tifffile as tiff
from tqdm import tqdm

# ==========================
# CONFIGURAÇÕES
# ==========================
TIF_PATH = r"D:\User data\InesMarques\202501024\fish6\c2downscaledbacksubtraction.tif"  # <-- muda aqui
SAVE_PATH = TIF_PATH.replace(".tif", "_ZattenuationCorrected_Cellpose.tif")

REFERENCE = "median"   # "median" ou "max"
SMOOTHING = 5           # suavizar curva de atenuação ao longo do Z

# ==========================
# FUNÇÃO PRINCIPAL
# ==========================

def smooth_curve(vec, w=5):
    """Aplicar smoothing 1D (janela média)."""
    return np.convolve(vec, np.ones(w)/w, mode='same')

def correct_z_attenuation(stack):
    """Normaliza brilho entre slices profundas e superficiais."""
    z = stack.shape[0]

    print("📊 Calculando intensidade média de cada slice...")
    means = np.array([np.mean(stack[i]) for i in tqdm(range(z))], dtype=np.float32)

    # Suavizar perfil de atenuação
    means_smooth = smooth_curve(means, SMOOTHING)

    # Referência
    if REFERENCE == "median":
        ref_value = np.median(means_smooth)
    else:
        ref_value = np.max(means_smooth)

    print(f"🎯 Intensidade alvo: {ref_value:.4f}")

    corrected = np.zeros_like(stack, dtype=np.float32)

    print("🔧 A corrigir intensidade Z...")
    for i in tqdm(range(z)):
        factor = ref_value / (means_smooth[i] + 1e-6)
        corrected[i] = stack[i] * factor

    return corrected

# ==========================
# PIPELINE
# ==========================

print("📥 A carregar TIFF 3D...")
img = tiff.imread(TIF_PATH).astype(np.float32)

if img.ndim != 3:
    raise ValueError("A imagem deve ser 3D (Z, Y, X).")

print("🔧 A aplicar correção de atenuação axial...")
corrected = correct_z_attenuation(img)

print("✂️ A aplicar clipping de outliers...")
p_low = np.percentile(corrected, 0.1)
p_high = np.percentile(corrected, 99.9)
corrected = np.clip(corrected, p_low, p_high)

print("📏 Normalizando para 16-bit...")
corrected -= corrected.min()
corrected /= corrected.max()
corrected_16 = (corrected * 65535).astype(np.uint16)

print(f"💾 A guardar: {SAVE_PATH}")
tiff.imwrite(SAVE_PATH, corrected_16)

print("✅ FEITO — stack corrigido para Cellpose 3D")


📥 A carregar TIFF 3D...
🔧 A aplicar correção de atenuação axial...
📊 Calculando intensidade média de cada slice...


100%|██████████| 614/614 [00:09<00:00, 62.02it/s]


🎯 Intensidade alvo: 112.0123
🔧 A corrigir intensidade Z...


100%|██████████| 614/614 [00:30<00:00, 19.81it/s]


✂️ A aplicar clipping de outliers...
📏 Normalizando para 16-bit...
💾 A guardar: D:\User data\InesMarques\202501024\fish6\c2downscaledbacksubtraction_ZattenuationCorrected_Cellpose.tif
✅ FEITO — stack corrigido para Cellpose 3D


In [3]:
import numpy as np
import tifffile as tiff
from tqdm import tqdm

# ==========================
# CONFIG
# ==========================
TIF_PATH = r"D:\User data\InesMarques\202501024\fish6\c2downscaledbacksubtraction.tif"
SAVE_PATH = TIF_PATH.replace(".tif", "_ZattenuationCorrected_Cellpose2.tif")

REFERENCE = "median"   # "median" ou "max"
SMOOTHING = 5

# ==========================
# FUNÇÕES
# ==========================

def smooth_curve(vec, w=5):
    return np.convolve(vec, np.ones(w)/w, mode='same')

def correct_z_attenuation(stack):
    z = stack.shape[0]
    dtype = stack.dtype  # guardar o tipo original (ex: uint16)

    print("📊 Calculando intensidade média de cada slice...")
    means = np.array([np.mean(stack[i]) for i in tqdm(range(z))], dtype=np.float32)
    means_smooth = smooth_curve(means, SMOOTHING)

    # referência
    if REFERENCE == "median":
        ref_value = np.median(means_smooth)
    else:
        ref_value = np.max(means_smooth)

    print(f"🎯 Intensidade alvo: {ref_value:.4f}")

    corrected = np.zeros_like(stack, dtype=np.float32)

    print("🔧 A aplicar correção multiplicativa...")
    for i in tqdm(range(z)):
        factor = ref_value / (means_smooth[i] + 1e-6)
        corrected[i] = stack[i] * factor

    # ⚠ SEM NORMALIZAR, SEM CLIPPING
    # Voltar para o mesmo tipo (ex: uint16), mas mantendo o intervalo original
    corrected = np.clip(corrected, 0, np.iinfo(dtype).max)
    corrected = corrected.astype(dtype)

    return corrected

# ==========================
# PIPELINE
# ==========================

print("📥 A carregar TIFF...")
img = tiff.imread(TIF_PATH)

if img.ndim != 3:
    raise ValueError("A imagem deve ser 3D (Z, Y, X).")

print("🔧 A corrigir atenuação axial...")
corrected = correct_z_attenuation(img)

print(f"💾 A guardar: {SAVE_PATH}")
tiff.imwrite(SAVE_PATH, corrected, imagej=True)

print("✅ FEITO — preservando histogram, bit-depth e aparência no Fiji")


📥 A carregar TIFF...
🔧 A corrigir atenuação axial...
📊 Calculando intensidade média de cada slice...


100%|██████████| 614/614 [00:14<00:00, 41.41it/s]


🎯 Intensidade alvo: 112.0122
🔧 A aplicar correção multiplicativa...


100%|██████████| 614/614 [01:13<00:00,  8.36it/s]


💾 A guardar: D:\User data\InesMarques\202501024\fish6\c2downscaledbacksubtraction_ZattenuationCorrected_Cellpose2.tif
✅ FEITO — preservando histogram, bit-depth e aparência no Fiji


C:\Users\ABBE User\anaconda520\envs\cellpose_env\lib\site-packages\tifffile\tifffile.py:3801: UserWarning: <tifffile.TiffWriter 'c2downscaledbac…ed_Cellpose2.tif'> truncating ImageJ file
  warnings.warn(
